In [ ]:
# Install the exact Mazinger source bundled in this video dataset
import os
import subprocess
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient
try:
    OPENAI_API_KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise RuntimeError("secret value is empty")
except Exception as secret_error:
    secret_message = "Kaggle Secret OPENAI_API_KEY is missing or not attached. Add it under this notebook's Add-ons > Secrets."
    os.makedirs("/kaggle/working", exist_ok=True)
    with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as error_file:
        error_file.write(secret_message + "\n")
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as report_file:
        json.dump({"Status": "Failed", "Target_Language": "Hindi", "Output_File": "Missing"}, report_file)
    raise RuntimeError(secret_message) from secret_error
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_MODEL"] = "qwen/qwen3.8-27b"
mazinger_candidates = list(Path("/kaggle/input").rglob("pyproject.toml"))
mazinger_candidates = [p.parent for p in mazinger_candidates if p.parent.name.lower() == "mazinger"]
if not mazinger_candidates:
    raise FileNotFoundError("Bundled Mazinger source is missing from the Kaggle dataset. Re-upload the dataset containing mazinger/pyproject.toml.")
mazinger_path = str(mazinger_candidates[0])
print(f"Installing bundled Mazinger source: {mazinger_path}")
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "--no-cache-dir", f"{mazinger_path}[all]"], check=True)
subprocess.run(["python", "-m", "pip", "install", "--no-cache-dir", "faster-whisper", "demucs", "omnivoice"], check=True)


In [ ]:
# Task 2: Find the uploaded video in Kaggle input directory
import os
import glob
import shutil
import json

input_dir = "/kaggle/input"
video_path = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith(('.mp4', '.mkv', '.avi', '.mov')):
            video_path = os.path.join(root, file)
            break
    if video_path:
        break

if not video_path:
    raise FileNotFoundError("Video not found in Kaggle input directory!")
print(f"Found video at: {video_path}")

In [ ]:
# Execute Mazinger Pipeline wrapped in Try-Except
import traceback
import subprocess

try:
    print("\U0001f680 Starting Mazinger Dubbing Pipeline...")
    base_dir = "/kaggle/working/mazinger_output"
    os.makedirs(base_dir, exist_ok=True)
    
    # First notebook cell loads the secret; never embed it in the notebook or CLI args.
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("local model key was not loaded from Kaggle Secret OPENAI_API_KEY. Rerun notebook from the first cell.")
    print("LLM provider: local model OpenAI-compatible endpoint; key loaded from Kaggle Secret.")
    # Run ASR and TTS on Kaggle CUDA; use local model only for LLM text/image calls.
    import sys
    cmd = [
        sys.executable, "-m", "mazinger", "dub", video_path, 
        "--target-language", "Hindi", 
        "--output-type", "video",
        "--device", "cuda",
        "--transcribe-method", "faster-whisper",
        "--openai-base-url", os.environ["OPENAI_BASE_URL"],
        "--llm-model", os.environ["OPENAI_MODEL"]
    ]
    
    print(f"Executing: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=base_dir, capture_output=True, text=True)
    
    print(f"STDOUT:\n{result.stdout}")
    print(f"STDERR:\n{result.stderr}")
    
    if result.returncode != 0:
        raise Exception(f"Mazinger CLI failed with return code {result.returncode}")
    
    print("\u2705 Mazinger finished successfully.")
    
    # Copy final video and report to /kaggle/working/
    output_candidates = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower() == 'dubbed.mp4':
                output_candidates.append(os.path.join(root, file))
    output_candidates = [p for p in output_candidates if os.path.getsize(p) > 0]
    output_video = output_candidates[-1] if output_candidates else None
    
    if output_video:
        shutil.copy(output_video, "/kaggle/working/output.mp4")
        print("Copied final video to /kaggle/working/output.mp4")
    else:
        err_msg = "\u26a0\ufe0f Could not find output.mp4 from Mazinger!"
        print(err_msg)
        with open("/kaggle/working/error_log.txt", "w") as f:
            f.write(err_msg + "\n")
        
    # Generate report.json
    report_data = {
        "Status": "Success" if output_video else "Failed",
        "Target_Language": "Hindi",
        "Output_File": os.path.basename(output_video) if output_video else "Missing",
        "WER": "Not measured",
        "Sync_Offset": "Not measured",
        "Speaker_Similarity": "Not measured"
    }
    with open("/kaggle/working/report.json", "w") as f:
        json.dump(report_data, f)
    if not output_video:
        raise RuntimeError("Mazinger exited without a non-empty dubbed.mp4 output; inspect error_log.txt.")
        
except Exception:
    error_text = traceback.format_exc()
    print(f"\u274c Error occurred!\n{error_text}")
    with open("/kaggle/working/error_log.txt", "w", encoding="utf-8") as f:
        f.write(error_text)
    with open("/kaggle/working/report.json", "w", encoding="utf-8") as f:
        json.dump({"Status": "Failed", "Target_Language": "Hindi", "Output_File": "Missing", "WER": "Not measured", "Sync_Offset": "Not measured", "Speaker_Similarity": "Not measured"}, f)
    raise
